# درس ۰۳ - الگوهای طراحی عامل

در این درس، ما سه الگوی پایه‌ای طراحی برای ساخت عوامل هوش مصنوعی مؤثر را بررسی می‌کنیم:

۱. **دستورالعمل‌های واضح برای عامل** — ساخت درخواست‌های دقیق و نقش‌محور که رفتار عامل را هدایت می‌کنند  
۲. **خروجی ساخت‌یافته با مدل‌های Pydantic** — تضمین بازگشت داده‌های قابل پیش‌بینی و اعتبارسنجی شده توسط عامل‌ها  
۳. **عوامل با مسئولیت واحد** — طراحی عوامل متمرکز که هر کدام یک کار را به خوبی انجام می‌دهند

ما هر الگو را روی یک سناریوی **پیشنهاددهنده مقصد سفر** اعمال خواهیم کرد، به تدریج سیستمی می‌سازیم که بتواند مقصدها را پیشنهاد کند، در دسترس بودن را بررسی کند و امور لجستیکی را مدیریت نماید.


## راه‌اندازی


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## الگوی ۱: دستورالعمل‌های روشن برای نماینده

تأثیرگذارترین الگو همچنین ساده‌ترین است: نوشتن دستورالعمل‌های واضح و دقیق برای نماینده خود.

دستورالعمل‌های خوب تعریف می‌کنند:
- **چه کسی** نماینده است (شخصیت و لحن)
- **چه کاری** باید انجام دهد (مسئولیت‌ها به صورت گام به گام)
- **چگونه** باید رفتار کند (محدودیت‌ها و سبک)

در زیر، ما یک نماینده مشاور سفر با دستورالعمل‌های صریح ایجاد می‌کنیم که هر پاسخی که ارائه می‌دهد را شکل می‌دهد.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## الگوی ۲: خروجی ساختارمند با مدل‌های پایدانتیک

متن آزاد برای گفتگو مفید است، اما سیستم‌های پایین‌دست به داده‌های ساختارمند نیاز دارند.
با جفت کردن **مدل‌های پایدانتیک** با یک **تابع ابزار**، ما می‌توانیم:

- یک طرحواره دقیق برای خروجی عامل تعریف کنیم  
- پاسخ‌ها را به‌صورت خودکار اعتبارسنجی کنیم  
- نتایج عامل را به‌طور قابل اعتماد در منطق برنامه ادغام کنیم  

همچنین ابزاری معرفی می‌کنیم که جزئیات مقصد را بازمی‌گرداند تا عامل توصیه‌های خود را بر اساس داده‌های واقعی پایه‌گذاری کند.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## الگوی ۳: عامل‌های با مسئولیت واحد

کارهای پیچیده با تقسیم کار بین چند عامل متمرکز که هر کدام مسئولیتی واحد دارند، بهتر انجام می‌شوند:

- یک **کارشناس مقصد** که درباره مکان‌ها و در دسترس بودن آن‌ها اطلاع دارد  
- یک **برنامه‌ریز لجستیک** که پروازها، هتل‌ها و برنامه‌های سفر را مدیریت می‌کند

این الگو منعکس‌کننده اصل مهندسی نرم‌افزار «تفکیک نگرانی‌ها» است — هر عامل را می‌توان به‌طور مستقل آسان‌تر آزمایش، نگهداری و بهبود داد.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## خلاصه

در این درس، سه الگوی طراحی عاملیت‌دار را در یک سناریوی توصیه‌گر سفر اعمال کردیم:

| الگو | ایده کلیدی | مزیت |
|---|---|---|
| **دستورات واضح** | تعریف شخصیت، مسئولیت‌ها و محدودیت‌ها از ابتدا | رفتار عامل ثابت و هماهنگ با برند |
| **خروجی ساختاریافته** | استفاده از مدل‌های Pydantic به عنوان قالب پاسخ | نتایج معتبر و قابل خواندن توسط ماشین |
| **مسئولیت واحد** | دادن یک وظیفه متمرکز به هر عامل | آسان‌تر برای تست، نگهداری و ترکیب |

این الگوها به طور طبیعی ترکیب می‌شوند — می‌توانید دستورات واضح را با خروجی ساختاریافته در داخل یک عامل با مسئولیت واحد ترکیب کنید تا سیستم‌های مقاوم و آماده تولید بسازید.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسؤولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان بومی خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، توصیه می‌شود از ترجمه حرفه‌ای انسانی استفاده شود. ما در قبال سوءتفاهم‌ها یا تفسیرهای نادرستی که ناشی از استفاده از این ترجمه باشد، مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
